In [ ]:
%env ENV_FOR_DYNACONF = prod
%env DYNACONF_GIT_CHECKOUT = feature/B-2895893

In [ ]:
import ltv_helpers.non_spark_helpers as nsh
import ltv_helpers.pipeline_helpers as ph
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib import cm
from pyspark.sql import functions as F
from specialty_ltv.config.paths import paths as p

In [ ]:
pd.options.display.max_columns = 500

FEATURE_BRANCH = "feature/B-2895893"
V1_4_RELEASE = "NB_SPL_v1.4.0"
FEATURE_LABEL = "Feature"
V1_4_LABEL = "v1.4.0"


def to_v1_4(path):
    return path.replace(FEATURE_BRANCH, V1_4_RELEASE)

In [ ]:
def plot_counts_with_two_lines(
    df,
    group_col,
    count_col,
    value_cols,
    line_labels=None,
    group_order=None,
    figsize=None,
    title=None,
    xlabel=None,
    ylabel_left="Count of Policies",
    ylabel_right=None,
    rotate_xticks=False,
):
    if isinstance(value_cols, str):
        value_cols = [value_cols]
    assert len(value_cols) == 2, "value_cols must be a list of two column names"

    line_styles = ["-", "-"]
    colors = ["blue", "orange"]
    markers = ["o", "o"]

    grouped = (
        df.groupby(group_col, dropna=False)
        .agg(
            count=(count_col, "count"),
            value1=(value_cols[0], "mean"),
            value2=(value_cols[1], "mean"),
        )
        .reset_index()
    )

    if figsize is None:
        figsize = (max(8, 0.28 * len(grouped)), 5)

    if group_order:
        grouped[group_col] = pd.Categorical(
            grouped[group_col], categories=group_order, ordered=True
        )
        grouped = grouped.sort_values(group_col)

    x = range(len(grouped[group_col]))

    fig, ax1 = plt.subplots(figsize=figsize)
    ax1.bar(x, grouped["count"], width=0.6, label="Count")
    ax1.set_xlabel(xlabel if xlabel else group_col)
    ax1.set_ylabel(ylabel_left)
    ax1.set_xticks(x)
    if rotate_xticks:
        ax1.set_xticklabels(grouped[group_col], rotation=90)
    else:
        ax1.set_xticklabels(grouped[group_col])

    ax2 = ax1.twinx()
    for idx, col in enumerate(["value1", "value2"]):
        label = line_labels[idx] if line_labels else f"Average {value_cols[idx]}"
        ax2.plot(
            x,
            grouped[col],
            color=colors[idx],
            marker=markers[idx],
            linestyle=line_styles[idx],
            label=label,
        )
    ax2.set_ylim(bottom=0)
    if not ylabel_right:
        ylabel_right = f"Average {', '.join(value_cols)}"
    ax2.set_ylabel(ylabel_right)

    ax1.legend(loc="upper left")
    ax2.legend(loc="upper right")

    plt.title(title or f"Count and Averages by {group_col}")
    plt.tight_layout()
    plt.show()

In [ ]:
def make_waterfall(df, components, title_input):
    components_order = list(components.keys())
    missing = [c for c in components_order if c not in df.index]
    assert not missing, f"components not found in df: {missing}"

    labels = [components[c][0] for c in components_order]
    signs = np.array([components[c][1] for c in components_order])
    values = np.array([df[c] for c in components_order], dtype=float)

    bar_heights = [values[0]]
    for i, c in enumerate(components_order[1:-1], 1):
        bar_heights.append(values[i] * signs[i])
    bar_heights.append(values[-1])

    bottoms = [0]
    cumulative = values[0]
    for h in bar_heights[1:-1]:
        bottoms.append(cumulative)
        cumulative += h
    bottoms.append(0)

    residual = cumulative - bar_heights[-1]
    if abs(residual) > 0.01:
        print(f"WARNING [{title_input}] waterfall does not close, residual={residual:,.4f}")

    fig, ax = plt.subplots(figsize=(12, 6))
    colors = cm.tab20(np.arange(len(bar_heights)))
    ax.bar(labels, bar_heights, bottom=bottoms, color=colors)

    for i, (b, h) in enumerate(zip(bottoms, bar_heights)):
        ax.text(
            i, b + h, f"{h:.0f}", ha="center", va="bottom", fontsize=9, color="black"
        )

    ax.axhline(
        bar_heights[0] + sum(bar_heights[1:-1]),
        color="red",
        linestyle="--",
        label="Final $",
    )
    plt.xticks(rotation=45, ha="right")
    plt.ylabel("Value")
    plt.title(title_input)
    plt.legend()
    plt.tight_layout()
    plt.show()

In [ ]:
def check_policy_exp(id):
    df = public_feature.filter((F.col("adw_pol_id") == id)).select(sel_col).toPandas()
    for item in ["service", "overhead", "claims", "commission"]:
        ratio_new_name = f"expense_ratio_{item}_new"
        ratio_renew_name = f"expense_ratio_{item}_renew"
        exp_name = f"lifetime_exp_{item}"
        exp_recalc = (
            df["premium_new"] * df[ratio_new_name]
            + df["premium_renew"] * df[ratio_renew_name]
        )
        assert df[exp_name][0] == exp_recalc[0]
    for item in ["acquisition", "marketing"]:
        ratio_new_name = f"expense_ratio_{item}_new"
        ratio_renew_name = f"expense_ratio_{item}_renew"
        exp_name = f"lifetime_exp_{item}"
        yearly_discount = (1 + 0.03) / (1 + 0.072)
        new_prem = df["premium_less_reinsurance"][0] * (yearly_discount**0.5)
        exp_recalc = (
            new_prem * df[ratio_new_name] + df["premium_renew"] * df[ratio_renew_name]
        )
        assert round(df[exp_name][0], 2) == round(exp_recalc[0], 2)
    for col in check_col:
        assert (
            df[col].iloc[0]
            == df_exp_2026.loc[df_exp_2026["channel"] == df.chnl_bnd[0], col].iloc[0]
        )

In [ ]:
from ltv_helpers.spark import create_spark

spark = create_spark(buckets=p.buckets, app_name="lr_spl_")

# I. Load Scores and merge data

In [ ]:
cols_to_keep = [
    "ply_policy_id",
    "ple",
    "ltv",
    "cac_x_mkt",
    "aac_mkt",
    "drv_full_premium_amt",
    "lifetime_premium",
    "lifetime_loss",
    "e006scl_claims_exp",
    "e006scl_acquisition_exp",
    "e006scl_lifetime_exp",
    "e006scl_commission_exp_renew",
    "e006scl_commission_exp_new",
    "cost_of_capital",
    "cost_of_capital_20pct",
    "cat_loss_amt",
    "balance_amt",
    "investment_income",
    "e006scl_tax",
]

group_cols = ["release_day", "ply_pt_state_cd", "drv_chnl_of_bnd"]

feature_cols_to_keep = cols_to_keep + group_cols

In [ ]:
df_feature = nsh.read_parquet_s3_to_pandas(
    p.score_internal_results, columns=feature_cols_to_keep
)
df_v1_4 = nsh.read_parquet_s3_to_pandas(
    to_v1_4(p.score_internal_results), columns=cols_to_keep
)

In [ ]:
dropped = {
    "feature": sorted(set(feature_cols_to_keep) - set(df_feature.columns)),
    "v1_4": sorted(set(cols_to_keep) - set(df_v1_4.columns)),
}
assert not any(dropped.values()), dropped
print(df_feature.shape, df_v1_4.shape)

In [ ]:
df_v1_4_select = df_v1_4.rename(
    columns={c: f"{c}_v1_4" for c in df_v1_4.columns if c != "ply_policy_id"}
)

combined_df = df_feature.merge(df_v1_4_select, how="inner", on=["ply_policy_id"])
combined_df["release_mth_yr"] = combined_df["release_day"].astype(str).str[:7]

assert len(combined_df) == len(df_feature), (len(combined_df), len(df_feature))
combined_df.head(2)

# II. Univariate Plots

In [ ]:
group_dict = {
    "drv_chnl_of_bnd": "Channel of Bind",
    "ply_pt_state_cd": "State",
    "release_mth_yr": "Release Month",
}


def univar_plots(metric, ylabel_right, metric_label):
    for group_col, group_name in group_dict.items():
        plot_counts_with_two_lines(
            df=combined_df,
            group_col=group_col,
            count_col="ply_policy_id",
            value_cols=[metric, f"{metric}_v1_4"],
            line_labels=[
                f"SPL Renters {metric_label} {FEATURE_LABEL}",
                f"SPL Renters {metric_label} {V1_4_LABEL}",
            ],
            group_order=None,
            title=f"SPL Renters {metric_label}: {V1_4_LABEL} vs {FEATURE_LABEL}",
            xlabel=group_name,
            ylabel_right=ylabel_right,
            rotate_xticks=(
                True if group_col in ["ply_pt_state_cd", "release_mth_yr"] else False
            ),
        )

## a. LTV

In [ ]:
univar_plots("ltv", "Average ($)", "LTV")

## b. PLE

In [ ]:
univar_plots("ple", "PLE (years)", "PLE")

## c. AAC

In [ ]:
univar_plots("aac_mkt", "Average AAC ($)", "AAC")

# III. Waterfall Plots

In [ ]:
combined_df.columns = combined_df.columns.str.replace(
    r"^e006scl_", "", regex=True
)

wf_cols = [
    "lifetime_premium",
    "lifetime_loss",
    "balance_amt",
    "cat_loss_amt",
    "commission_exp_new",
    "commission_exp_renew",
    "investment_income",
    "tax",
    "ple",
    "ltv",
    "cost_of_capital",
    "cost_of_capital_20pct",
    "acquisition_exp",
    "claims_exp",
    "lifetime_exp",
    "cac_x_mkt",
    "aac_mkt",
]

for col in wf_cols:
    combined_df[f"{col}_diff"] = combined_df[f"{col}_v1_4"] - combined_df[col]

In [ ]:
ltv_components = {
    "ltv_v1_4": (f"SPL Renters {V1_4_LABEL} LTV", 1),
    "lifetime_premium_diff": ("Lifetime Premium Diff", -1),
    "lifetime_loss_diff": ("Lifetime Loss Diff", 1),
    "balance_amt_diff": ("Balance Diff", 1),
    "cat_loss_amt_diff": ("Cat Loss Diff", 1),
    "claims_exp_diff": ("Claims Expense Diff", 1),
    "lifetime_exp_diff": ("Lifetime Expense Diff", 1),
    "commission_exp_renew_diff": ("Commission Renew Diff", 1),
    "cost_of_capital_diff": ("Cost of Capital Diff", 1),
    "tax_diff": ("Tax Diff", 1),
    "investment_income_diff": ("Investment Income Diff", -1),
    "ltv": (f"SPL Renters {FEATURE_LABEL} LTV", 0),
}

aac_components = {
    "aac_mkt_v1_4": (f"SPL Renters {V1_4_LABEL} AAC", 1),
    "lifetime_premium_diff": ("Lifetime Premium Diff", -1),
    "lifetime_loss_diff": ("Lifetime Loss Diff", 1),
    "balance_amt_diff": ("Balance Diff", 1),
    "cat_loss_amt_diff": ("Cat Loss Diff", 1),
    "claims_exp_diff": ("Claims Expense Diff", 1),
    "lifetime_exp_diff": ("Lifetime Expense Diff", 1),
    "commission_exp_renew_diff": ("Commission Renew Diff", 1),
    "cost_of_capital_20pct_diff": ("Cost of Capital (AAC, 20%) Diff", 1),
    "tax_diff": ("Tax Diff", 1),
    "investment_income_diff": ("Investment Income Diff", -1),
    "commission_exp_new_diff": ("Commission New Diff", 1),
    "acquisition_exp_diff": ("Acquisition Expense Diff", 1),
    "aac_mkt": (f"SPL Renters {FEATURE_LABEL} AAC", 0),
}

## 1. Overall Charts

In [ ]:
wf_value_cols = [
    col for col in combined_df.columns if col.startswith(tuple(wf_cols))
]
overall_compare_averages = combined_df[wf_value_cols].mean()

### a. LTV

In [ ]:
make_waterfall(
    df=overall_compare_averages,
    components=ltv_components,
    title_input=f"Overall LTV Comparison of SPL Renters {FEATURE_LABEL} and {V1_4_LABEL}",
)

### b. AAC

In [ ]:
make_waterfall(
    df=overall_compare_averages,
    components=aac_components,
    title_input=f"Overall AAC Comparison of SPL Renters {FEATURE_LABEL} and {V1_4_LABEL}",
)

## 2. By Channel Chart

In [ ]:
channel_compare_averages = (
    combined_df.groupby("drv_chnl_of_bnd")[wf_value_cols].mean().reset_index()
)

### a. LTV Comparison

In [ ]:
for channel in channel_compare_averages.drv_chnl_of_bnd:
    make_waterfall(
        df=channel_compare_averages.loc[
            channel_compare_averages["drv_chnl_of_bnd"] == channel
        ].iloc[0],
        components=ltv_components,
        title_input=f"LTV Comparison of SPL Renters {FEATURE_LABEL} and {V1_4_LABEL} -- {channel}",
    )

### b. AAC Comparison

In [ ]:
for channel in channel_compare_averages.drv_chnl_of_bnd:
    make_waterfall(
        df=channel_compare_averages.loc[
            channel_compare_averages["drv_chnl_of_bnd"] == channel
        ].iloc[0],
        components=aac_components,
        title_input=f"AAC Comparison of SPL Renters {FEATURE_LABEL} and {V1_4_LABEL} -- {channel}",
    )

# IV. Check Balance

In [ ]:
bal_feat = nsh.read_parquet_s3_to_pandas(p.balance_factors)
bal_benchmark = nsh.read_parquet_s3_to_pandas(to_v1_4(p.balance_factors))

In [ ]:
round(bal_benchmark, 3)

In [ ]:
bal_feat = round(bal_feat, 3)
bal_feat["calculated_original_lr"] = (
    bal_feat["lr_x_cat_target"] - bal_feat["bal_factor"]
)
bal_feat

In [ ]:
df_agg_for_bal_pd = nsh.read_parquet_s3_to_pandas(p.agg_for_balance)

In [ ]:
assert bal_feat["calculated_original_lr"][0] == round(
    df_agg_for_bal_pd["lr_non_cat"][0], 3
)

# V. Check Expenses

In [ ]:
df_exp_2026 = nsh.read_parquet_s3_to_pandas(p.processed_expense_all)
df_exp_2026.columns = [
    col.replace("_e006scl", "").replace("_lifetime", "_service")
    for col in df_exp_2026.columns
]

In [ ]:
public_feature_path = (
    "tmx-smsiweb/specialty-ltv/prod/release/"
    + FEATURE_BRANCH
    + "/ltv_calc/public_results/"
)

internal_feature = ph.read_parquet_s3(spark, p.score_internal_results)
public_feature = ph.read_parquet_s3(spark, public_feature_path)
public_feature = public_feature.join(
    internal_feature.select(
        "ply_policy_id", "premium_less_reinsurance", "premium_new", "premium_renew"
    ),
    on=[public_feature.adw_pol_id == internal_feature.ply_policy_id],
)
public_feature.select("adw_pol_id").show(3)

In [ ]:
sel_col = [
    "line",
    "adw_pol_id",
    "release_date",
    "chnl_bnd",
    "written_premium",
    "premium_new",
    "premium_renew",
    "premium_less_reinsurance",
    "expense_ratio_acquisition_new",
    "expense_ratio_acquisition_renew",
    "expense_ratio_commission_new",
    "expense_ratio_commission_renew",
    "expense_ratio_service_new",
    "expense_ratio_service_renew",
    "expense_ratio_marketing_new",
    "expense_ratio_marketing_renew",
    "expense_ratio_overhead_new",
    "expense_ratio_overhead_renew",
    "expense_ratio_claims_new",
    "expense_ratio_claims_renew",
    "lifetime_exp_marketing",
    "lifetime_exp_service",
    "lifetime_exp_acquisition",
    "lifetime_exp_overhead",
    "lifetime_exp_claims",
    "lifetime_exp_commission",
    "lifetime_exp_commission_new",
    "lifetime_exp_commission_renew",
]

check_col = [
    "expense_ratio_acquisition_new",
    "expense_ratio_acquisition_renew",
    "expense_ratio_commission_new",
    "expense_ratio_commission_renew",
    "expense_ratio_service_new",
    "expense_ratio_service_renew",
    "expense_ratio_marketing_new",
    "expense_ratio_marketing_renew",
    "expense_ratio_overhead_new",
    "expense_ratio_overhead_renew",
    "expense_ratio_claims_new",
    "expense_ratio_claims_renew",
]

In [ ]:
check_policy_exp(10000001499695)

In [ ]:
check_policy_exp(10000001225111)

In [ ]:
check_policy_exp(10000001229980)